# Detectarea textelor umane și AI în limba română

Experiment v3: 7.950 de texte, Qwen3.5-4B și LoRA.

Datele se descarcă automat din repository. Rezultatele și checkpointurile se salvează în propriul Google Drive.
Începe cu `RUN_MODE="prepare"` până la secțiunea 5, apoi pilot până la 7. Repornește sesiunea între faze.
Pentru antrenare setează `RUN_MODE="train"` și `CONFIRM_FULL_TRAINING=True`. Testul din secțiunea 9 se activează explicit după calibrare.
Folosește același RUN_NAME între faze. Pentru reluare/calibrare indică numele rulării existente.
Scorurile istorice sunt în rezultate/. O rulare nouă poate avea rezultate diferite.


## Setup
### 1. Configurație
Rulare curată după OOM: `RUN_MODE = "prepare"`, `MAX_SEQ_LENGTH = 12288`. Selectați GPU în Colab înainte de instalare.
Încărcați ZIP-ul în Drive, `MyDrive/SDCGIA/colab/`. Nu îl dezarhivați manual.
Pentru pilot/antrenare schimbați modul și reporniți sesiunea, apoi executați în ordine.
Un RUN_NAME nou înseamnă un experiment nou; nu suprascrieți rezultatele finale existente.


In [ ]:
from pathlib import Path
import re

OFFLINE_VALIDATION = False  # Numai companionul local de verificare setează True.
RUN_MODE = "prepare"  # "prepare", "pilot", "train", "calibrate", "evaluate"
RUN_NAME = "qwen35_4b_ro_full_v3_" + __import__("datetime").datetime.now(__import__("datetime").timezone.utc).strftime("%Y%m%d_%H%M%S")
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 12288
SEED = 3407
TRAIN_BATCH_SIZE = 1
ENABLE_T4_KV_FIX = True
GRADIENT_ACCUMULATION = 16
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
CONFIRM_FULL_TRAINING = False  # True numai după pilot, în modul train.
RESUME_CHECKPOINT = None  # Calea completă către checkpoint-N din ACELAȘI experiment.

BUNDLE_PATH = Path("/content/drive/MyDrive/SDCGIA/colab/SDCGIA_date_complete_v3.zip")
EXTRACT_ROOT = Path("/content/sdcgia_full_v3")
SAVE_ROOT = Path("/content/drive/MyDrive/SDCGIA/experiments")
EXPECTED_MANIFEST_SHA = 'b982d32417193eb41beef98dd2880a896816c796dd54ef9dfaf0140a3e6ae948'
assert RUN_MODE in {"prepare", "pilot", "train", "calibrate", "evaluate"}
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME)
assert MAX_SEQ_LENGTH >= 512
print("Mod:", RUN_MODE, "| Model:", MODEL_NAME, "| Context:", MAX_SEQ_LENGTH)


### 2. Google Drive și GPU
Autorizați montarea propriului Drive. Dacă GPU-ul nu este disponibil, opriți-vă aici;
nu continuați antrenarea pe CPU. Nu există acces automat al agentului la contul vostru.


In [ ]:
if not OFFLINE_VALIDATION:
    from google.colab import drive
    drive.mount("/content/drive")
    import subprocess
    import shutil
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Selectează Runtime > Change runtime type > T4 GPU, apoi reia sesiunea.")
    check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True)
    if check.returncode:
        raise RuntimeError("GPU NVIDIA indisponibil. Runtime > Change runtime type > GPU.")
    print(check.stdout)
else:
    print("OFFLINE: Drive și verificarea GPU nu sunt executate.")


### 3. Instalare — celula originală
Poate dura câteva minute. Dacă Colab cere restart după instalare, reporniți sesiunea și reluați de la configurație.
Nu este necesar să instalați aceste biblioteci pe calculatorul local.


In [ ]:
if not OFFLINE_VALIDATION:
    installation_result = get_ipython().run_cell('!pip install -q "torch==2.8.0" "torchvision==0.23.0" "torchaudio==2.8.0" "transformers==5.2.0" "unsloth==2026.10.7" "unsloth_zoo==2026.10.7" "trl==0.22.2"')
    if installation_result.error_before_exec or installation_result.error_in_exec:
        raise RuntimeError("Instalarea a eșuat; nu continua.")
else:
    print("OFFLINE: instalarea este omisă.")


## Steps
### 4. Verificarea pachetului și a partițiilor
Se verifică hash-urile înainte de încărcarea helperului. Metadatele sunt păstrate separat:
modelul primește numai articolul și întrebarea fixă; eticheta `gold` este ținta de învățare.


In [ ]:
import json, hashlib, zipfile, types
import urllib.request
DATA_URL = "https://raw.githubusercontent.com/Neculai13/SDCGIA-v3/main/date/SDCGIA_date_complete_v3.zip"
EXPECTED_BUNDLE_SHA = "8e6b5c02fe685fd620f6b81d652f00be814e19c526350be6cb175411d2253adb"
BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not BUNDLE_PATH.exists():
    temporary_bundle = BUNDLE_PATH.with_suffix(".download")
    urllib.request.urlretrieve(DATA_URL, temporary_bundle)
    if hashlib.sha256(temporary_bundle.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA:
        raise RuntimeError("Arhiva descărcată nu corespunde datelor experimentului.")
    temporary_bundle.replace(BUNDLE_PATH)
if hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA:
    raise RuntimeError("Arhiva locală diferă de corpusul v3. Nu o suprascriu.")
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    manifest_bytes = archive.read("bundle_manifest.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == EXPECTED_MANIFEST_SHA, "Pachet greșit sau modificat"
    bundle_manifest = json.loads(manifest_bytes)
    runtime_bytes = archive.read("colab_human_ai_runtime.py")
    assert hashlib.sha256(runtime_bytes).hexdigest() == bundle_manifest["files"]["colab_human_ai_runtime.py"]
runtime = types.ModuleType("sdcgia_runtime")
exec(compile(runtime_bytes, "colab_human_ai_runtime.py", "exec"), runtime.__dict__)
runtime.unpack_verified(BUNDLE_PATH, EXTRACT_ROOT, EXPECTED_MANIFEST_SHA)
data, metadata = runtime.load_splits(EXTRACT_ROOT)
assert {k: len(v) for k, v in data.items()} == bundle_manifest["counts"]
for split in data:
    from collections import Counter
    print(split, len(data[split]), dict(Counter(m["label"] for m in metadata[split])))
print("Verificări de integritate și separare: OK. Textele nu au fost modificate.")


### 5. Tokenizer și auditul lungimilor — fără antrenare
Se numără tokenii întregului input formatat (text + întrebare + opțiuni), nu cuvintele.
Niciun articol nu este eliminat sau scurtat. Raportul este salvat în Drive.
**Dacă există depășiri, trimiteți raportul înainte să treceți la pilot.** Putem mări contextul,
dar consumul de memorie și durata cresc; nu presupunem că toate textele vor încăpea pe T4.


In [ ]:
if not OFFLINE_VALIDATION:
    import unsloth  # înainte de transformers/torch
    from transformers import AutoTokenizer
    from unsloth.models.clef import encode_record
    tokenizer_audit = AutoTokenizer.from_pretrained(MODEL_NAME)
    def audit_encode(row, budget):
        return encode_record(tokenizer_audit, row, max_length=budget)
    length_report = runtime.inspect_lengths(data, metadata, audit_encode, MAX_SEQ_LENGTH)
    analysis_dir = SAVE_ROOT / RUN_NAME / ("analysis_ctx" + str(MAX_SEQ_LENGTH))
    runtime.save_json(analysis_dir / "token_lengths.json", length_report)
    print(json.dumps({k: v for k, v in length_report.items() if k != "records"}, indent=2))
    CONTEXT_OK = length_report["over_context"] == 0
    print("Context suficient:", CONTEXT_OK)
    if not CONTEXT_OK:
        print("OPRIRE DE SIGURANȚĂ: pilotul/antrenarea sunt blocate, nu se acceptă trunchiere implicită.")
else:
    CONTEXT_OK = False
    print("OFFLINE: tokenizarea reală nu este verificată; nu există rezultat GPU sau de lungime în tokeni.")


### 5.1. Corecție atenție T4 — GQA explicit
Opțională prin `ENABLE_T4_KV_FIX`. Restrânsă la versiunile diagnosticate, T4, key/value float16 cu 4 capete de dimensiune 256 și fără mască explicită.
Dezactivează selectorul GQA nativ numai pentru acest caz. Transformers execută propria replicare KV; nu se schimbă weights, dtype, scale, dropout sau cauzalitatea.
Patch-ul trăiește numai în sesiunea Python; nu rescrie bibliotecile. Există `attention_fix.restore()`, dar pentru alt experiment folosiți o sesiune nouă.
Testul sintetic forward/backward cu 64 tokeni a trecut pe T4 conform ieșirii furnizate de utilizator; nu certifică pilotul lung.
Profilerul pilotului urmărește operatorii executați; nu păstrează tensorii sau stack-urile. Nu există fallback silențios acceptat în criteriul de succes.


In [ ]:
"""Narrow, reversible Transformers 5.2 / PyTorch 2.8 T4 GQA workaround.

No imports of torch/transformers at import, no disk changes, no weight changes.
The original HF sdpa_attention_forward performs repeat_kv when this selector
returns False. Its mask/dropout/scale/causal handling stays unchanged.
"""
import hashlib
import inspect
from pathlib import Path

POLICY = 't4_explicit_kv_transformers_selector_v1'
EXPECTED_VERSIONS = {'torch': '2.8.0', 'transformers': '5.2.0',
                     'unsloth': '2026.10.7', 'unsloth_zoo': '2026.10.7'}


class T4ExplicitKVFix:
    def __init__(self, torch_module, sdpa_module, versions):
        for name, expected in EXPECTED_VERSIONS.items():
            actual = versions.get(name, '').split('+')[0]
            if actual != expected:
                raise RuntimeError(f'Runtime nevalidat pentru corecție: {name}={actual}; așteptat {expected}.')
        if tuple(torch_module.cuda.get_device_capability(0)) != (7, 5):
            raise RuntimeError('Corecția este restrânsă la GPU Turing (compute capability 7.5).')
        original = getattr(sdpa_module, 'use_gqa_in_sdpa', None)
        if not callable(original) or not callable(getattr(sdpa_module, 'repeat_kv', None)):
            raise RuntimeError('API Transformers diferit; corecția nu a fost aplicată.')
        if hasattr(original, '_sdcgia_fix'):
            raise RuntimeError('Corecția este deja aplicată. Repornește sesiunea înainte de rerulare.')
        if list(inspect.signature(original).parameters) != ['attention_mask', 'key']:
            raise RuntimeError('Semnătura selectorului GQA diferă de cea verificată.')
        self.torch, self.module, self.original = torch_module, sdpa_module, original
        self.calls, self.shapes = 0, set()
        self.versions = {k: versions[k] for k in EXPECTED_VERSIONS}
        try:
            source = Path(inspect.getsourcefile(original)).read_bytes()
            self.source_sha256 = hashlib.sha256(source).hexdigest()
        except (OSError, TypeError):
            self.source_sha256 = None

        def selector(attention_mask, key):
            targeted = (attention_mask is None and key.device.type == 'cuda'
                        and key.ndim == 4 and key.shape[1] == 4 and key.shape[-1] == 256
                        and key.dtype == self.torch.float16
                        and tuple(self.torch.cuda.get_device_capability(key.device)) == (7, 5))
            if targeted:
                self.calls += 1
                if len(self.shapes) < 32:
                    self.shapes.add(tuple(int(n) for n in key.shape))
                return False
            return self.original(attention_mask, key)

        selector._sdcgia_fix = self
        self.selector = selector
        self.module.use_gqa_in_sdpa = selector

    def assert_installed(self):
        if self.module.use_gqa_in_sdpa is not self.selector:
            raise RuntimeError('Selectorul de atenție a fost înlocuit ulterior; nu continua pilotul.')

    def reset_counters(self):
        self.calls, self.shapes = 0, set()

    def config(self):
        return {'policy': POLICY, 'scope': 'T4 CUDA float16 K/V heads=4 head_dim=256 mask=None',
                'versions': self.versions, 'transformers_sdpa_source_sha256': self.source_sha256,
                'model_weights_changed': False, 'dtype_changed': False, 'article_text_changed': False,
                'selector': 'explicit_repeat_kv_via_existing_transformers_path'}

    def snapshot(self):
        return {**self.config(), 'bypassed_native_gqa_calls': self.calls,
                'observed_key_shapes_before_repeat': [list(s) for s in sorted(self.shapes)]}

    def restore(self):
        self.assert_installed()
        self.module.use_gqa_in_sdpa = self.original


def inspect_attention_events(events):
    """Inspect executed ATen operators, not merely available/enabled backend flags."""
    operators = {event.key: int(event.count) for event in events
                 if 'scaled_dot_product' in event.key.lower()}
    forward = operators.get('aten::_scaled_dot_product_efficient_attention', 0)
    backward = operators.get('aten::_scaled_dot_product_efficient_attention_backward', 0)
    math_calls = sum(n for key, n in operators.items()
                     if key.startswith('aten::') and 'scaled_dot_product_attention_math' in key)
    return {'operators': operators, 'efficient_forward_calls': forward,
            'efficient_backward_calls': backward, 'math_sdpa_calls': math_calls,
            'efficient_forward_and_backward_observed': forward > 0 and backward > 0,
            'no_math_sdpa_observed': math_calls == 0,
            'scope': 'CPU profiler observes dispatched ATen operators during the two real pilot steps; no tensor shapes/stacks retained.'}


def require_efficient_pilot(report, bypassed_calls):
    if bypassed_calls <= 0:
        raise RuntimeError('Corecția nu a fost traversată. Pilotul nu este confirmat.')
    if not report['efficient_forward_and_backward_observed'] or not report['no_math_sdpa_observed']:
        raise RuntimeError('Profilerul nu confirmă ruta memory-efficient fără SDPA math. Trimite raportul; nu porni antrenarea completă.')


In [ ]:
ATTENTION_FIX_SOURCE_SHA = '237106bbac04388e8a126a5e78331b5edc7f0e1b6d8756a4c9992633911aad12'
attention_fix = None
if not OFFLINE_VALIDATION and RUN_MODE != "prepare" and ENABLE_T4_KV_FIX:
    if MODEL_NAME != "unsloth/Qwen3.5-4B":
        raise RuntimeError("Această corecție este verificată doar pentru Qwen3.5-4B/T4; nu schimba modelul aici.")
    import torch, importlib, importlib.metadata as package_metadata
    sdpa_module = importlib.import_module("transformers.integrations.sdpa_attention")
    patch_versions = {name: package_metadata.version(name) for name in EXPECTED_VERSIONS}
    attention_fix = T4ExplicitKVFix(torch, sdpa_module, patch_versions)
    print("Corecție activă:", attention_fix.config())
else:
    print("Corecția nu este activată în prepare/offline sau este dezactivată explicit.")


### 6. Încărcarea modelului și conversia strictă
Se execută numai în pilot/train/evaluate și numai dacă toate textele încap integral.
Pilotul este un experiment separat; pentru train se repornește sesiunea și se pornește din modelul de bază.
Orice rând omis sau input scurtat de bibliotecă oprește execuția înainte de antrenare.


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE != "prepare":
    if not CONTEXT_OK:
        raise RuntimeError("Context insuficient. Trimite raportul; nu rula antrenarea cu texte tăiate.")
    if RUN_MODE == "train" and not CONFIRM_FULL_TRAINING:
        raise RuntimeError("După pilot, setează CONFIRM_FULL_TRAINING=True explicit.")
    if "model" in globals():
        raise RuntimeError("Repornește sesiunea înainte de încărcarea repetată a modelului.")
    from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
    from transformers import set_seed
    import torch
    assert torch.cuda.is_available()
    set_seed(SEED)
    phase = "pilot_" + RUN_NAME if RUN_MODE == "pilot" else RUN_NAME
    RUN_DIR = SAVE_ROOT / phase
    if RUN_MODE == "train" and (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea este deja încheiată. Folosește calibrate/evaluate sau un RUN_NAME nou.")
    if RUN_MODE == "pilot" and (RUN_DIR / "pilot_complete.json").exists():
        raise RuntimeError("Pilotul este deja încheiat. Folosește un RUN_NAME nou pentru alt pilot.")
    if RUN_MODE == "train" and RESUME_CHECKPOINT is None and list((RUN_DIR / "checkpoints").glob("checkpoint-*")):
        raise RuntimeError("Există checkpointuri. Specifică RESUME_CHECKPOINT sau folosește un RUN_NAME nou.")
    load_path = str(RUN_DIR / "model_calibrated") if RUN_MODE == "evaluate" else str(RUN_DIR / "model_uncalibrated") if RUN_MODE == "calibrate" else MODEL_NAME
    if RUN_MODE == "evaluate" and not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Lipsește modelul calibrat al acestui experiment.")
    if RUN_MODE == "calibrate" and not (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea completă nu este salvată; reia checkpointul în modul train.")
    model, tokenizer = FastDecisionModel.from_pretrained(model_name=load_path,
        max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    if RUN_MODE in {"pilot", "train"}:
        model = FastDecisionModel.get_peft_model(model, r=16, lora_alpha=16,
            lora_dropout=0, use_gradient_checkpointing="unsloth", random_state=SEED)
    assert getattr(model, "is_clef", False), "API schimbat: acest notebook este verificat pentru Clef/Qwen3.5"
    items = runtime.strict_items(data, metadata, tokenizer, model, FastDecisionModel, length_report)
    run_config = {"model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH,
        "seed": SEED, "batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation": 1 if RUN_MODE == "pilot" else GRADIENT_ACCUMULATION,
        "epochs": NUM_TRAIN_EPOCHS, "learning_rate": LEARNING_RATE,
        "lora_rank": 16, "bundle_manifest_sha256": EXPECTED_MANIFEST_SHA,
        "phase": "pilot" if RUN_MODE == "pilot" else "full", "no_truncation": True}
    import importlib.metadata as package_metadata, sys, inspect
    from unsloth.models import decision as decision_module
    versions = {}
    for package in ("unsloth", "unsloth_zoo", "torch", "transformers", "peft", "trl", "bitsandbytes"):
        try: versions[package] = package_metadata.version(package)
        except package_metadata.PackageNotFoundError: versions[package] = "unknown"
    run_config["runtime_versions"] = versions
    run_config["decision_source_sha256"] = runtime.sha_bytes(Path(inspect.getfile(decision_module)).read_bytes())
    run_config["base_revision"] = model.decision_config.get("base_revision")
    run_config["attention_fix"] = attention_fix.config() if attention_fix else {"policy": "original"}
    run_config["attention_fix_source_sha256"] = ATTENTION_FIX_SOURCE_SHA
    runtime.freeze_run(RUN_DIR, run_config)
    runtime.save_json(RUN_DIR / ("environment_" + RUN_MODE + ".json"), {
        "versions": versions, "python": sys.version, "gpu": torch.cuda.get_device_name(0),
        "decision_source_sha256": run_config["decision_source_sha256"],
        "decision_config": model.decision_config})
    print("Conversie integrală OK:", {k: len(v) for k, v in items.items()})
else:
    print("Încărcarea modelului și antrenarea sunt inactive în modul prepare/offline.")


### 7. Pilot sau antrenare completă
Pilot: exact două etape pe cele mai lungi două articole din train. Profilerul urmărește operațiile ATen pentru a verifica ruta memory-efficient la forward și backward. Durata include overhead de profiling.
Nu este un rezultat final și nu folosește testul. Train: o epocă pe toate cele 5557 texte.
Nu folosim calibration/test pentru alegerea checkpointului. Checkpointurile se scriu direct în Drive la 10 etape,
se păstrează ultimele două cu starea optimizerului. Întreruperea poate pierde progresul de după ultimul checkpoint.


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"pilot", "train"}:
    from transformers import TrainingArguments
    import time
    if RUN_MODE == "pilot":
        selected = sorted(range(len(items["train"])), key=lambda i: len(items["train"][i]["input_ids"]))
        indexes = selected[-2:]
        training_items = [items["train"][i] for i in indexes]
    else:
        training_items = items["train"]
    if RESUME_CHECKPOINT is not None:
        checkpoint = Path(RESUME_CHECKPOINT).resolve()
        if checkpoint.parent != (RUN_DIR / "checkpoints").resolve() or not (checkpoint / "trainer_state.json").exists():
            raise RuntimeError("Checkpointul nu aparține acestui experiment sau nu conține starea trainerului.")
    arguments = TrainingArguments(output_dir=str(RUN_DIR / "checkpoints"),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=1 if RUN_MODE == "pilot" else GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_TRAIN_EPOCHS, max_steps=2 if RUN_MODE == "pilot" else -1,
        warmup_steps=0 if RUN_MODE == "pilot" else 10, learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine", weight_decay=.01,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        eval_strategy="no", logging_steps=1, save_strategy="steps", save_steps=50,
        save_total_limit=2, save_only_model=False, report_to="none", seed=SEED)
    trainer = DecisionTrainer(model=model, processing_class=tokenizer,
        train_dataset=training_items, args=arguments)
    if attention_fix:
        attention_fix.assert_installed()
        attention_fix.reset_counters()
    if RUN_MODE == "pilot":
        assert TRAIN_BATCH_SIZE == 1 and len(training_items) == 2
        pilot_selection = [{"id": metadata["train"][i]["id"],
                            "tokens": len(items["train"][i]["input_ids"])} for i in indexes]
        runtime.save_json(RUN_DIR / "pilot_selection.json", pilot_selection)
        print("Articole pilot:", pilot_selection)
    torch.cuda.reset_peak_memory_stats()
    started = time.monotonic()
    from contextlib import nullcontext
    from torch.profiler import profile, ProfilerActivity
    profiling = profile(activities=[ProfilerActivity.CPU], record_shapes=False,
                        profile_memory=False, with_stack=False) if RUN_MODE == "pilot" else nullcontext()
    try:
        with profiling as attention_profile:
            training_stats = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
        torch.cuda.synchronize()
        if RUN_MODE == "pilot":
            backend_report = inspect_attention_events(attention_profile.key_averages())
            backend_report["attention_fix"] = attention_fix.snapshot() if attention_fix else None
            runtime.save_json(RUN_DIR / "attention_backend_report.json", backend_report)
            print(json.dumps(backend_report, indent=2))
            if attention_fix:
                require_efficient_pilot(backend_report, attention_fix.calls)
    except Exception as pilot_error:
        runtime.save_json(RUN_DIR / ("pilot_failed.json" if RUN_MODE == "pilot" else "training_failed.json"), {
            "exception_type": type(pilot_error).__name__, "message": str(pilot_error),
            "attention_fix": attention_fix.snapshot() if attention_fix else None,
            "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3})
        raise
    training_seconds = time.monotonic()-started
    if not all(torch.isfinite(parameter).all().item() for parameter in model.parameters() if parameter.requires_grad):
        raise RuntimeError("Parametri antrenabili ne-finiți după pilot/antrenare; nu se marchează complet.")
    model.save_pretrained(str(RUN_DIR / "model_uncalibrated"))
    completed = {"attention_fix": attention_fix.snapshot() if attention_fix else None,
                 "mode": RUN_MODE, "training_metrics": training_stats.metrics,
                 "seconds": training_seconds,
                 "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3}
    runtime.save_json(RUN_DIR / ("pilot_complete.json" if RUN_MODE == "pilot" else "training_complete.json"), completed)
    print(json.dumps(completed, indent=2))
    if RUN_MODE == "pilot":
        print("Pilot terminat. NU continua spre evaluarea finală. Repornește sesiunea înainte de train.")
else:
    print("Antrenarea nu este activată.")


### 8. Calibrare și salvarea modelului final
Numai după antrenarea completă: calibrăm probabilitățile pe cele 1194 exemple dedicate, cu batch 1.
Pragul rămâne 0,5; nu îl optimizăm pe test. Se salvează adapterul LoRA împreună cu decision head și configurația.


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"train", "calibrate"}:
    if not (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea completă nu este confirmată.")
    calibration_metrics = FastDecisionModel.calibrate(model, tokenizer, items["calibration"], batch_size=1)
    runtime.save_json(RUN_DIR / "calibration_metrics.json", calibration_metrics)
    model.save_pretrained(str(RUN_DIR / "model_calibrated"))
    saved_files = list((RUN_DIR / "model_calibrated").iterdir())
    assert any(p.name.startswith("adapter_model") for p in saved_files)
    assert (RUN_DIR / "model_calibrated" / "joint_head.safetensors").exists(), "Decision head lipsă"
    final_fingerprint = runtime.adapter_fingerprint(RUN_DIR / "model_calibrated")
    runtime.save_json(RUN_DIR / "calibration_complete.json", {"adapter_fingerprint": final_fingerprint})
    print("Model calibrat salvat în:", RUN_DIR / "model_calibrated")
elif not OFFLINE_VALIDATION and RUN_MODE == "evaluate":
    final_fingerprint = runtime.adapter_fingerprint(RUN_DIR / "model_calibrated")
    saved = json.loads((RUN_DIR / "calibration_complete.json").read_text())
    assert saved["adapter_fingerprint"] == final_fingerprint
    print("Model calibrat reîncărcat; configurația și adapterul sunt verificate.")
else:
    print("Calibrarea nu este activată.")


## Checks
### 9. Test final — confirmare explicită
Păstrați False până când configurația experimentului este fixată. Pentru evaluare setați True numai aici.
Predicțiile sunt salvate după fiecare articol și pot fi reluate pentru același model și aceleași date.
Nu modificați hiperparametrii pe baza rezultatelor testului și nu prezentați pilotul ca rezultat final.
FPR = proporția textelor human clasificate greșit ca AI. ECE folosește 10 intervale egale de încredere.
Rezultatele per generator folosesc textele AI și articolele human pereche; unele subgrupuri sunt mici.


In [ ]:
RUN_FINAL_TEST = False  # Schimbă în True numai după antrenare și calibrare.
if not OFFLINE_VALIDATION and RUN_FINAL_TEST:
    if RUN_MODE not in {"train", "calibrate", "evaluate"} or not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Este necesar modelul final calibrat, nu pilotul.")
    FastDecisionModel.for_inference(model)
    predictions = runtime.predict_saved(data["test"], metadata["test"],
        lambda state, questions: FastDecisionModel.predict(model, tokenizer, state, questions),
        RUN_DIR / "test_predictions.jsonl", final_fingerprint + EXPECTED_MANIFEST_SHA)
    test_results = runtime.grouped_metrics(predictions)
    runtime.save_json(RUN_DIR / "test_metrics.json", test_results)
    print(json.dumps(test_results, ensure_ascii=False, indent=2))
else:
    print("Testul final nu este activat. Modelul salvat și checkpointurile nu sunt pierdute.")


### 10. Arhivă de siguranță
Arhiva include adapterul final și rapoartele, nu checkpointurile/optimizerul (acestea rămân separat în Drive).
Se verifică integritatea ZIP și hash-urile fișierelor modelului. Aceasta nu este încă o probă de reîncărcare pe GPU;
modul evaluate verifică separat că adapterul poate fi reîncărcat și folosit pentru predicții.


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"train", "calibrate", "evaluate"}:
    if not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Modelul final nu este încă salvat.")
    import zipfile, hashlib
    archive_path = RUN_DIR / "human_ai_final.zip"
    temporary = archive_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_DIR.rglob("*")):
            relative = path.relative_to(RUN_DIR)
            if path.is_file() and "checkpoints" not in relative.parts and path not in {archive_path, temporary}:
                archive.write(path, relative.as_posix())
    with zipfile.ZipFile(temporary) as archive:
        assert archive.testzip() is None
        for path in (RUN_DIR / "model_calibrated").rglob("*"):
            if path.is_file():
                assert archive.read(path.relative_to(RUN_DIR).as_posix()) == path.read_bytes()
    temporary.replace(archive_path)
    print("Salvare verificată:", archive_path)
    print("Dimensiune MB:", round(archive_path.stat().st_size/1024**2, 2))
else:
    print("Nu există încă un model final de arhivat.")


## Next Steps
- În v2, contextul 12288 acoperă maximul de 10465 tokeni raportat; auditul se repetă și blochează orice tăiere.
- După pilot trimiteți durata și memoria maximă; acestea nu oferă încă o estimare exactă pentru întreaga epocă.
- După antrenare păstrați modelul calibrat, `test_metrics.json`, `test_predictions.jsonl`, configurația și ZIP-ul final.
- Reluare antrenare: aceeași configurație, RUN_MODE=train, CONFIRM_FULL_TRAINING=True și RESUME_CHECKPOINT din Drive.
- Reluare evaluare: aceeași configurație, RUN_MODE=evaluate, fără reantrenare. Activați testul numai dacă e necesar.
- Dacă antrenarea s-a salvat dar calibrarea a fost întreruptă, folosiți RUN_MODE=calibrate, fără reantrenare.
- Dacă modificăm datele, schema sau hiperparametrii, folosim RUN_NAME nou; nu alterăm rezultatele vechi.

### Surse
- https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth
- Notebookul original local `Qwen3_5_(4B)-Decision.original.ipynb` (SHA în manifestul pachetului).
- Implementarea API verificată la pregătire: https://github.com/unslothai/unsloth/blob/main/unsloth/models/decision.py

Instalarea originală folosește versiuni actualizabile. Rapoartele de mediu păstrează versiunile și hash-ul implementării;
compatibilitatea exactă cu GPU-ul și versiunile Colab se confirmă prin pilot, nu prin verificarea locală.
